
### Overview
This notebook executes the foundational SQL warehouse layer and compares 6 attribution models:

**SQL Warehouse Schema** (Phase 1):
- `dim_channel` — Reference taxonomy of marketing channels
- `fact_sessions` — Chronological session fact table with `ROW_NUMBER()` sequencing
- `fact_conversions` — Converting sessions with real transaction revenue
- `fact_channel_transitions` — Markov transition matrix computed entirely in SQL
- `journey_paths` — Aggregated user touchpoint path strings

**Attribution Models** (Phase 2):
- Heuristic baselines: First-Click, Last-Click, Linear, Time-Decay
- Analytical Markov Chain: Closed-form $(I - Q)^{-1}$ matrix inversion
- Exact Full-Dataset Shapley Value: All $2^N$ coalitions enumerated without sampling


In [1]:
import os
import pandas as pd

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f'Working directory: {os.getcwd()}')
assert os.path.exists('scripts/01_build_warehouse.py'), "Run this notebook from the project root directory"
assert os.path.exists('data/raw/staging_sessions.csv'), "Run scripts/00_extract_bigquery.py first to extract raw data"
print("Project structure verified.")


Working directory: c:\Users\laksh kumar\Desktop\multi_touch_attribution
Project structure verified.


In [2]:
%run scripts/01_build_warehouse.py


Creating database: data/warehouse.db
Loading data/raw/staging_sessions.csv into staging_sessions table...
  Loaded 319,982 rows. NULL channel_grouping count: 0
Executing sql/01_dim_channel.sql...
Executing sql/02_fact_sessions.sql...
Executing sql/03_fact_conversions.sql...
Executing sql/04_fact_channel_transitions.sql...
Executing sql/05_journey_paths.sql...

  SECTION 5 VERIFICATION CHECKPOINTS
[fact_sessions]            Row count: 319,982 — MATCH
[fact_conversions]         Row count: 3,719
                           Total transactions: 3,851
                           Total revenue: $522,574.73
                           Real AOV: $135.70
[dim_channel]              Channels (8): ['(Other)', 'Affiliates', 'Direct', 'Display', 'Organic Search', 'Paid Search', 'Referral', 'Social']
[fact_channel_transitions] Total edges: 574,856

Top 10 Channel Transitions:
  from_channel     to_channel  transition_count
         Start Organic Search             99363
Organic Search           Null     


Run Attribution Models
Computes heuristic baselines, analytical Markov removal effects, and exact Shapley values across the full visitor cohort. Also generates the illustrative financial reconciliation layer and budget optimizer.

**Disclosure:** Spend figures are illustrative estimates, not sourced from a published benchmark. The GA4 sample dataset does not include actual advertising expenditure. AOV is computed directly from real revenue data.


In [3]:
%run scripts/02_run_attribution_models.py


Loading warehouse exports...
Real AOV (from fact_conversions): $135.70
  Total transactions: 3,851
  Total revenue: $522,574.73
Total Visitors: 258,650 | Converting Visitors: 3,341

Computing Heuristic Baselines (First-Click, Last-Click, Linear, Time-Decay)...
Computing Analytical Markov Chain via (I - Q)^-1 matrix inversion...
  Baseline absorption probability (Start -> Conversion): 0.012917
Computing Exact Full-Dataset Shapley Value...
  Enumerating 2^8 = 256 coalitions...

=== ATTRIBUTION MODEL COMPARISON ===
       Channel  First_Click  Last_Click  Linear  Time_Decay  Markov  Shapley
      Referral       1352.0      1509.0 1431.11     1436.43 1419.49  1420.67
Organic Search       1034.0      1022.0 1028.90     1028.31 1052.01  1033.00
        Direct        697.0       562.0  623.54      618.96  591.41   629.17
   Paid Search        174.0       162.0  168.26      168.04  166.65   169.50
       Display         53.0        54.0   57.54       57.70   57.74    56.50
        Social      


Load and inspect the model comparison table produced above.


In [4]:
df_models = pd.read_csv('data/exports/model_comparison.csv')
print("=== ATTRIBUTION MODEL COMPARISON ===")
display(df_models)

# Shapley vs Last-Click delta
df_models['Shap_vs_LC_Delta'] = (df_models['Shapley'] - df_models['Last_Click']).round(2)
df_models['Shap_vs_LC_Pct'] = ((df_models['Shapley'] - df_models['Last_Click']) / df_models['Last_Click'] * 100).round(1)
print("\n=== SHAPLEY vs LAST-CLICK DELTA ===")
display(df_models[['Channel', 'Last_Click', 'Shapley', 'Shap_vs_LC_Delta', 'Shap_vs_LC_Pct']])


=== ATTRIBUTION MODEL COMPARISON ===


,Channel,First_Click,Last_Click,Linear,Time_Decay,Markov,Shapley
0,Referral,1352.0,1509.0,1431.11,1436.43,1419.49,1420.67
1,Organic Search,1034.0,1022.0,1028.90,1028.31,1052.01,1033.00
2,Direct,697.0,562.0,623.54,618.96,591.41,629.17
3,Paid Search,174.0,162.0,168.26,168.04,166.65,169.50
4,Display,53.0,54.0,57.54,57.70,57.74,56.50
5,Social,30.0,31.0,30.66,30.56,40.17,31.17
6,Affiliates,1.0,1.0,1.00,1.00,13.45,1.00
7,(Other),0.0,0.0,0.00,0.00,0.07,0.00



=== SHAPLEY vs LAST-CLICK DELTA ===


,Channel,Last_Click,Shapley,Shap_vs_LC_Delta,Shap_vs_LC_Pct
0,Referral,1509.0,1420.67,-88.33,-5.9
1,Organic Search,1022.0,1033.00,11.00,1.1
2,Direct,562.0,629.17,67.17,12.0
3,Paid Search,162.0,169.50,7.50,4.6
4,Display,54.0,56.50,2.50,4.6
5,Social,31.0,31.17,0.17,0.5
6,Affiliates,1.0,1.00,0.00,0.0
7,(Other),0.0,0.00,0.00,NaN


---
## Observations
Interpret the output above. Key patterns to look for:
- Which channels gain or lose credit when moving from Last-Click to Shapley?
- Does the Markov removal effect agree directionally with Shapley, or diverge?
- Are there channels with large heuristic credit but small algorithmic credit (or vice versa)?

Proceed to **Notebook 02** for financial reconciliation visualizations, the budget optimizer, and the Markov Chain network graph.
